In [15]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd
from prophet import Prophet
import matplotlib.pyplot as plt

In [6]:
rental_long_df = pd.read_csv("dataset/processed/rental_long.csv")
rental_long_df

,RegionID,RegionName,StateName,SizeRank,Date,Rent_Index
0,394913,"New York, NY",NY,1,2015-01-31,2434.921089
1,753899,"Los Angeles, CA",CA,2,2015-01-31,1831.131781
2,394463,"Chicago, IL",IL,3,2015-01-31,1486.135985
3,394514,"Dallas, TX",TX,4,2015-01-31,1129.748436
4,394692,"Houston, TX",TX,5,2015-01-31,1275.474390
...,...,...,...,...,...,...
82903,394996,"Portales, NM",NM,915,2025-06-30,904.814815
82904,394805,"Los Alamos, NM",NM,916,2025-06-30,2300.000000
82905,786253,"Brownsville, TN",TN,921,2025-06-30,1150.000000
82906,394342,"Atchison, KS",KS,929,2025-06-30,1211.250000


In [17]:
def validate_forecast_clean(df, target_col, cutoff="2022-12-31", plot_example=False):
    """
    df: long-format DataFrame with columns Date, RegionID, RegionName, StateName, and target_col.
    target_col: name of the value column to forecast.
    cutoff: last training date (string YYYY-MM-DD).
    plot_example: if True, shows a plot for the first region to help debug.
    """
    results = []
    first_plot_done = False

    for region_id, group in df.groupby("RegionID"):
        try:
            metro = group[["Date", target_col]].rename(
                columns={"Date": "ds", target_col: "y"}
            ).dropna()

            # Split train/test
            train = metro[metro["ds"] <= cutoff]
            test = metro[metro["ds"] > cutoff].copy()

            if len(train) < 24 or len(test) == 0:
                continue  # need at least 2 years of training and some test points

            # Prophet
            model = Prophet(yearly_seasonality=True, weekly_seasonality=False)
            model.fit(train)

            future = model.make_future_dataframe(periods=len(test), freq="M")
            forecast = model.predict(future)
            forecast_test = forecast.set_index("ds").loc[test["ds"]]

            # Drop rows with 0 or missing actuals to avoid div-by-zero in MAPE
            valid_mask = (test["y"] > 0) & (~test["y"].isna())
            test = test.loc[valid_mask]
            forecast_test = forecast_test.loc[valid_mask]

            if len(test) == 0:
                continue

            mae = mean_absolute_error(test["y"], forecast_test["yhat"])
            rmse = np.sqrt(mean_squared_error(test["y"], forecast_test["yhat"]))
            mape = np.mean(np.abs((test["y"] - forecast_test["yhat"]) / test["y"])) * 100

            results.append({
                "RegionID": region_id,
                "RegionName": group["RegionName"].iloc[0],
                "StateName": group["StateName"].iloc[0],
                "Target": target_col,
                "MAE": round(mae, 2),
                "RMSE": round(rmse, 2),
                "MAPE": round(mape, 2)
            })

            # Optional one-time plot for visual check
            if plot_example and not first_plot_done:
                import matplotlib.pyplot as plt
                plt.figure(figsize=(10, 5))
                plt.plot(train["ds"], train["y"], label="Train")
                plt.plot(test["ds"], test["y"], label="Actual")
                plt.plot(forecast_test.index, forecast_test["yhat"], label="Forecast")
                plt.title(f"Example: {group['RegionName'].iloc[0]} ({target_col})")
                plt.legend()
                plt.show()
                first_plot_done = True

        except Exception as e:
            print(f"Skipping RegionID {region_id}: {e}")

    return pd.DataFrame(results)

In [12]:
sliced_df = rental_long_df[rental_long_df["RegionName"] == "New York, NY"]
sliced_df

,RegionID,RegionName,StateName,SizeRank,Date,Rent_Index
0,394913,"New York, NY",NY,1,2015-01-31,2434.921089
658,394913,"New York, NY",NY,1,2015-02-28,2451.100181
1316,394913,"New York, NY",NY,1,2015-03-31,2465.526087
1974,394913,"New York, NY",NY,1,2015-04-30,2476.041728
2632,394913,"New York, NY",NY,1,2015-05-31,2480.672901
...,...,...,...,...,...,...
79618,394913,"New York, NY",NY,1,2025-02-28,3534.929495
80276,394913,"New York, NY",NY,1,2025-03-31,3546.124313
80934,394913,"New York, NY",NY,1,2025-04-30,3557.101368
81592,394913,"New York, NY",NY,1,2025-05-31,3567.011171


In [18]:
rent_validation = validate_forecast_clean(sliced_df, "Rent_Index")

10:29:11 - cmdstanpy - INFO - Chain [1] start processing
10:29:12 - cmdstanpy - INFO - Chain [1] done processing


Skipping RegionID 394913: Unalignable boolean Series provided as indexer (index of the boolean Series and of the indexed object do not match).


/home/avas/avas/zilliow-housing-research/.venv/lib/python3.12/site-packages/prophet/forecaster.py:1872: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  dates = pd.date_range(


In [21]:
rent_validation.head()

""
